# 02 - Visualize dataset and validate slot annotations
Loads the MetaPKLot `cars` and `spots` annotations (COCO-style JSON inside `.tar.xz`) for **PKLot and CNRPark-EXT** (PLds has annotations but no images in this mirror), and checks them against the images on disk:

1. Schema actually present (the dataset README differs from the files, so we trust the files).
2. Which annotation variant (`original` or `cropped`) matches the images on disk.
3. Car boxes and spot polygons are valid; occupied spots link to car annotations.
4. Whether spot polygons are fixed per camera (persistent slot ids).
5. Time structure: frames per day and gap between frames.
6. Overlay drawings for one image per camera.

**Inputs:** the MetaPKLot dataset. **Outputs:** `/kaggle/working/artifacts/annotations_v2/` (summary JSON, tables, overlay images, status). CPU is enough.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Step 2: configuration. The dataset folder is detected automatically.
DATASET_HINT = ""       # only needed if several datasets are attached: part of the right path, e.g. "metapklot"
DATA_ROOT = None        # or set the path yourself to skip detection
OUT = "/kaggle/working/artifacts/annotations_v2"

from src.data.discover import describe, find_dataset_root
if DATA_ROOT is None:
    DATA_ROOT = str(find_dataset_root("/kaggle/input", DATASET_HINT))
info = describe(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("files:", info["files"], "| top-level entries:", info["n_children"], info["children"])

In [ ]:
# Cell 3: load both annotation variants (this takes a minute)
import json, time
from pathlib import Path
from src.data import metapklot as mp
root = Path(DATA_ROOT)
t0 = time.time()
sets = {}
for variant in ("original", "cropped"):
    for kind in ("cars", "spots"):
        sets[(variant, kind)] = mp.load_set(root, variant, kind)
        s = sets[(variant, kind)]
        print(variant, kind, "| files:", len(s["schema"]), "| images:", len(s["images"]), "| annotations:", len(s["annotations"]))
print("schema (original/spots, first file):", json.dumps(sets[("original", "spots")]["schema"][0], indent=1)[:1500])
print("schema (original/cars, first file):", json.dumps(sets[("original", "cars")]["schema"][0], indent=1)[:1500])
print("loaded in", round(time.time() - t0, 1), "s")

In [ ]:
# Cell 4: which variant matches the real images? (all files checked for existence, 400 sampled for size)
checks = {}
for variant in ("original", "cropped"):
    imgs = sets[(variant, "spots")]["images"]
    checks[variant] = mp.check_images_on_disk(root, imgs, sample=400)
    print(variant, checks[variant])
# how do the two variants relate? same image, same spot, both coordinate sets
o, c = sets[("original", "spots")], sets[("cropped", "spots")]
oi = {i["image_id"]: i for i in o["images"]}
ci = {i["image_id"]: i for i in c["images"]}
first = o["annotations"][0]
same = [a for a in c["annotations"] if a["ann_id"] == first["ann_id"] and a["image_id"] == first["image_id"]]
print("original image:", {k: oi[first["image_id"]][k] for k in ("file_name", "width", "height", "rect")})
print("cropped  image:", {k: ci.get(first["image_id"], {}).get(k) for k in ("file_name", "width", "height", "rect")})
print("original spot bbox:", first["bbox"], "| cropped spot bbox:", same[0]["bbox"] if same else "not found")

In [ ]:
# Cell 5: validate cars and spots (original variant)
O_c, O_s = sets[("original", "cars")], sets[("original", "spots")]
val_cars = mp.validate_cars(O_c["images"], O_c["annotations"])
val_spots = mp.validate_spots(O_s["images"], O_s["annotations"], O_c["annotations"])
print("CARS:", json.dumps(val_cars, indent=1))
print("SPOTS:", json.dumps(val_spots, indent=1))
spot_imgs = {i["image_id"] for i in O_s["images"]}
car_imgs = {i["image_id"] for i in O_c["images"]}
print("images in spots json:", len(spot_imgs), "| in cars json:", len(car_imgs),
      "| cars-only:", len(car_imgs - spot_imgs), "| spots-only (no cars):", len(spot_imgs - car_imgs))

In [ ]:
# Cell 6: slot identity, time structure, climate per camera, day lists
ident = mp.slot_identity(O_s["images"], O_s["annotations"])
temporal = mp.temporal_structure(O_s["images"])
climates = mp.climate_counts(O_s["images"])
days = mp.read_day_lists(root)
print("SLOT IDENTITY (distinct polygons per camera vs spots per image):")
for k, v in ident.items():
    print(" ", k, v)
print("TIME (gap in minutes between consecutive frames of a day):")
print("  images without timestamp:", temporal["images_without_timestamp"])
for k, v in temporal["cameras"].items():
    print(" ", k, v)
print("CLIMATE:")
for k, v in climates.items():
    print(" ", k, v)
print("DAY LISTS:")
for k, v in days.items():
    print(" ", k, len(v), "entries, e.g.", v[:4])

In [ ]:
# Cell 7: overlays - one image per camera. Yellow = annotation rectangle, cyan = car boxes,
# green = empty spot, red = occupied spot.
from PIL import Image, ImageDraw
from IPython.display import display
os.makedirs(OUT + "/viz", exist_ok=True)
cars_by_img, spots_by_img = {}, {}
for a in O_c["annotations"]:
    cars_by_img.setdefault(a["image_id"], []).append(a)
for a in O_s["annotations"]:
    spots_by_img.setdefault(a["image_id"], []).append(a)
seen = set()
for img in sorted(O_s["images"], key=lambda i: i["file_name"]):
    if img["camera"] in seen:
        continue
    seen.add(img["camera"])
    im = Image.open(str(root / img["file_name"])).convert("RGB")
    d = ImageDraw.Draw(im)
    if img["rect"]:
        x, y, w, h = img["rect"]
        d.rectangle([x, y, x + w, y + h], outline=(255, 220, 0), width=3)
    for s in spots_by_img.get(img["image_id"], []):
        p = s["polygon"]
        if len(p) >= 6:
            d.polygon(list(zip(p[0::2], p[1::2])), outline=(220, 0, 0) if s["category_id"] == 1 else (0, 200, 0))
    for c in cars_by_img.get(img["image_id"], []):
        x, y, w, h = c["bbox"]
        d.rectangle([x, y, x + w, y + h], outline=(0, 220, 220), width=2)
    name = img["camera"].replace("/", "_") + ".jpg"
    im.save(OUT + "/viz/" + name)
    print(img["file_name"])
    display(im.resize((im.width // 2, im.height // 2)))

In [ ]:
# Cell 8: save tables and summary for notebook 03 (splits)
import csv, gzip
from src.common.status import new_run_id, write_status
with open(OUT + "/images.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["image_id", "file_name", "camera", "source", "width", "height", "rect",
                                      "timestamp", "day", "climate"], extrasaction="ignore")
    w.writeheader()
    for r in O_s["images"]:
        w.writerow(dict(r, rect=json.dumps(r["rect"])))
with gzip.open(OUT + "/spots.csv.gz", "wt", newline="") as f:
    w = csv.writer(f)
    w.writerow(["ann_id", "image_id", "car_id", "category_id", "bbox", "polygon"])
    for a in O_s["annotations"]:
        w.writerow([a["ann_id"], a["image_id"], a["car_id"], a["category_id"], json.dumps(a["bbox"]), json.dumps(a["polygon"])])
with gzip.open(OUT + "/cars.csv.gz", "wt", newline="") as f:
    w = csv.writer(f)
    w.writerow(["ann_id", "image_id", "car_id", "category_id", "bbox"])
    for a in O_c["annotations"]:
        w.writerow([a["ann_id"], a["image_id"], a["car_id"], a["category_id"], json.dumps(a["bbox"])])
summary = {"variant_check": checks, "cars": val_cars, "spots": val_spots, "slot_identity": ident,
           "temporal": temporal, "climates": climates, "day_lists": {k: len(v) for k, v in days.items()}}
with open(OUT + "/annotation_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
write_status(OUT + "/status.json", "02_visualize_and_validate", new_run_id("ann"), "COMPLETE")
print("saved to", OUT, sorted(os.listdir(OUT)))

## What to send back
Paste the output of cells 3 to 6, and say whether the overlays in cell 7 look right (spots on the parking spaces, red where a car is, cyan boxes on cars, everything inside the yellow rectangle). Then **Save Version** so `annotations_v2/` can be attached to notebook 03.